In [4]:
cd /mnt/team/rgud/pub/users/sunnypyl/for_Christian/

/mnt/team/rgud/pub/users/sunnypyl/for_Christian


/ihme/code/central_comp/miniconda/envs/2024-07-31T193602_py3.11/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
ls

NYHA1_baseline_draws_super_0806.xlsx  dependency_map_ppcm.csv
NYHA2_baseline_draws_super_0806.xlsx  duratuin_log_draws_super_0806.xlsx
NYHA3_baseline_draws_super_0806.xlsx  epi_ids_ppcm.csv
NYHA4_baseline_draws_super_0806.xlsx  severity_all_pred0802_comp.xlsx


In [13]:
import pandas as pd
import numpy as np

#"/mnt/team/rgud/pub/users/sunnypyl/for_Christian/"
# Reading the specific sheet 'SalesData' from the Excel file
df = pd.read_excel('/mnt/team/rgud/pub/users/sunnypyl/for_Christian/NYHA4_baseline_draws_super_0806.xlsx', sheet_name='Sheet 1')

# Define the logit to linear transformation function
def logit_to_linear(logit_mean):
    return 1.0 / (1.0 + np.exp(-logit_mean))

# Apply the transformation to numeric cells only, skipping non-numeric cells
def transform_if_numeric(x):
    try:
        return logit_to_linear(float(x))
    except (ValueError, TypeError):
        return x

transformed_df = df.applymap(transform_if_numeric)

# Output the transformed DataFrame to a new Excel file
transformed_file_path = 'NYHA4_baseline_draws_super_0806_linear.xlsx'
transformed_df.to_excel(transformed_file_path, index=False)

print(f"Transformation complete. The transformed data is saved in '{transformed_file_path}'.")


Transformation complete. The transformed data is saved in 'NYHA4_baseline_draws_super_0806_linear.xlsx'.


In [37]:
# Specify the folder containing your Excel files
folder_path = '/mnt/team/rgud/pub/users/sunnypyl/for_Christian/Central_Europe/'

# Specify the location name to filter
location_name = 'Latin America and Caribbean'

# Get a list of all Excel files in the folder
excel_files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

# Iterate over each file and process
for file in excel_files:
    file_path = os.path.join(folder_path, file)
    # Read the Excel file
    df = pd.read_excel(file_path)
    
    # Check if 'super_region_name' column exists
    if 'super_region_name' in df.columns:
        # Filter rows where 'super_region_name' matches the specified location
        filtered_df = df[df['super_region_name'] == location_name]
        
        # Save the filtered data to a new Excel file
        output_file_path = os.path.join(folder_path, f'{file}') #f'filtered_{file}'
        filtered_df.to_excel(output_file_path, index=False)
        print(f'Filtered data saved to {output_file_path}')
    else:
        print(f"'super_region_name' column not found in {file}")

print('Processing completed.')

Filtered data saved to /mnt/team/rgud/pub/users/sunnypyl/for_Christian/Central_Europe/NYHA1_baseline_draws_super_0806_linear.xlsx
Filtered data saved to /mnt/team/rgud/pub/users/sunnypyl/for_Christian/Central_Europe/NYHA2_baseline_draws_super_0806_linear.xlsx
Filtered data saved to /mnt/team/rgud/pub/users/sunnypyl/for_Christian/Central_Europe/NYHA3_baseline_draws_super_0806_linear.xlsx
Filtered data saved to /mnt/team/rgud/pub/users/sunnypyl/for_Christian/Central_Europe/NYHA4_baseline_draws_super_0806_linear.xlsx
Processing completed.


In [56]:
from __future__ import division
import pandas as pd
from get_draws.api import get_draws
from db_tools import ezfuncs as ez
import numpy as np
import sys
import os
from db_queries import (get_location_metadata, 
                        get_demographics,
                        get_covariate_estimates,
                        get_population)
import xlsxwriter as xl
import gbd.constants as gbd

release_id = 16


epi_demographics = get_demographics("epi", release_id=release_id)
most_detailed_ages = epi_demographics['age_group_id']
most_detailed_locs = epi_demographics['location_id']

# Convert list to DataFrame
list_df = pd.DataFrame(most_detailed_locs, columns=['location_id'])

location = get_location_metadata(location_set_id = 35, release_id = 16)
filtered_location = location[location['super_region_name'] == 'Latin America and Caribbean']
filtered_location = filtered_location[['location_id']]

# Merge DataFrames on 'location_id'
merged_df = pd.merge(list_df, filtered_location, on='location_id', how='right')

# Back to a list again
location_id_list = merged_df['location_id'].tolist()

#print(list_df)
#print(filtered_location)
print(merged_df)
print(location_id_list)


    location_id
0           103
1           120
2           121
3           122
4           123
..          ...
92         4773
93         4775
94         4774
95         4776
96          136

[97 rows x 1 columns]
[103, 120, 121, 122, 123, 104, 105, 106, 107, 108, 305, 109, 110, 111, 112, 113, 114, 115, 385, 393, 116, 117, 118, 119, 422, 124, 125, 126, 127, 128, 129, 130, 4643, 4644, 4645, 4646, 4649, 4650, 4647, 4648, 4652, 4653, 4654, 4655, 4656, 4657, 4651, 4658, 4659, 4660, 4661, 4662, 4663, 4664, 4665, 4666, 4667, 4668, 4669, 4670, 4671, 4672, 4673, 4674, 131, 132, 133, 134, 135, 4750, 4751, 4753, 4752, 4754, 4755, 4756, 4757, 4758, 4759, 4762, 4761, 4760, 4763, 4764, 4765, 4766, 4767, 4768, 4769, 4772, 4770, 4771, 4773, 4775, 4774, 4776, 136]


In [11]:
import pandas as pd
import numpy as np

# Define the parent directory path
parent_dir = '/mnt/team/rgud/pub/users/sunnypyl/for_Christian/Latin_America/'

# File names
file_names = {
    'control': 'NYHA1_baseline_draws_super_0806_linear.xlsx',
    'mild': 'NYHA2_baseline_draws_super_0806_linear.xlsx',
    'moderate': 'NYHA3_baseline_draws_super_0806_linear.xlsx',
    'severe': 'NYHA4_baseline_draws_super_0806_linear.xlsx'
}

# Dictionary to hold the NumPy arrays
data_dict = {}

for key, file_name in file_names.items():
    # Construct full file path
    file_path = os.path.join(parent_dir, file_name)
    
    # Read the Excel file into a Pandas DataFrame
    df = pd.read_excel(file_path)
    
    # Convert DataFrame to numeric only
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Convert the numeric DataFrame to a NumPy array
    data_dict[key] = numeric_df.to_numpy()

# Access the arrays
control = data_dict['control']
print(control)
mild = data_dict['mild']
moderate = data_dict['moderate']
severe = data_dict['severe']

[[0.25624379 0.243279   0.31593047 0.3413547  0.30264169 0.18493332
  0.2114516  0.19837052 0.42801996 0.36237418 0.08102768 0.139985
  0.19414485 0.63202378 0.60114605 0.24591619 0.42576655 0.20917019
  0.42542833 0.2850603  0.26048854 0.14220844 0.1026817  0.1420499
  0.12250299 0.34752628 0.3550731  0.33527257 0.03338711 0.22571698
  0.16263289 0.19771089 0.23132098 0.34906803 0.10684939 0.3684851
  0.37263097 0.0971684  0.12970709 0.08603011 0.48086947 0.46463567
  0.35150192 0.10707854 0.28255788 0.13046297 0.4110883  0.16933005
  0.2270008  0.32631296 0.19889108 0.52234399 0.1201796  0.46007325
  0.30675717 0.3230753  0.1018246  0.31310479 0.33876783 0.54379386
  0.19024439 0.52078714 0.15438364 0.17264408 0.22084079 0.22842449
  0.08690489 0.22841749 0.19032497 0.28908062 0.29832358 0.40733073
  0.11341218 0.33873569 0.05946391 0.23947314 0.46795645 0.55364697
  0.32049552 0.46990355 0.26232975 0.33162691 0.32139013 0.40596384
  0.31566049 0.3110114  0.43952805 0.27278894 0.3232

In [4]:
import os

# Define the directories
#dir1 = 'path/to/first/directory'
#dir2 = 'path/to/second/directory'
dir1 = '/mnt/share/scratch/users/chrish47/nonfatal_maternal/2024_08_12_12/27702/'
dir2 = '/mnt/share/scratch/users/chrish47/nonfatal_maternal/2024_07_16_20/27702/'

# List all files in each directory
files_dir1 = set(os.listdir(dir1))
files_dir2 = set(os.listdir(dir2))

# Find files unique to each directory
unique_to_dir1 = files_dir1 - files_dir2
unique_to_dir2 = files_dir2 - files_dir1

# Combine all unique files
files_not_in_both = unique_to_dir1.union(unique_to_dir2)

# Display the results
print("Files unique to first directory:", unique_to_dir1)
print("Files unique to second directory:", unique_to_dir2)
print("Files not present in both directories:", files_not_in_both)

# Paths to the two directories you want to compare
#directory1 = '/mnt/share/scratch/users/chrish47/nonfatal_maternal/2024_08_12_12/27702/'
#directory2 = '/mnt/share/scratch/users/chrish47/nonfatal_maternal/2024_07_16_20/27702/'

# Perform the comparison
#compare_directories(directory1, directory2)
#print("Done")

Files unique to first directory: set()
Files unique to second directory: set()
Files not present in both directories: set()
